# AçaIA — linha de base dos candidatos ao Copinho (Kaggle, GPU grátis)

Este notebook sobe cada modelo aberto com o **vLLM** (o mesmo servidor que vai rodar o Copinho na Fase 3) e roda o harness `ml/eval` nele. No fim, um modelo aberto maior faz o papel de **juiz** (rubrica e segurança).

**Antes de rodar:**
1. Conta no Kaggle com o **telefone verificado** (necessário para usar GPU).
2. *Settings* do notebook (painel à direita): **Accelerator → GPU T4 x2** e **Internet → On**.
3. Se algum modelo for "gated" no Hugging Face (pede para aceitar a licença), aceite no site e adicione o seu token em *Add-ons → Secrets* com o nome `HF_TOKEN`.
4. (Opcional) Para medir o Pro aqui também, adicione `PRO_API_KEY` em *Secrets*.

Cota grátis: ≈ 30 h de GPU por semana. Cada candidato no ENEM leva ≈ 10–20 min.

In [ ]:
# 1) Código do projeto (repositório público) e vLLM
!git clone --depth 1 https://github.com/eduardovicentinilevy/a-aia.git
!pip install -q vllm huggingface_hub
import vllm, sys
print("vLLM", vllm.__version__, "| Python", sys.version.split()[0])

In [ ]:
# 2) Segredos do Kaggle (opcional) -> variáveis de ambiente
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ("HF_TOKEN", "PRO_API_KEY"):
        try:
            os.environ[name] = secrets.get_secret(name)
            print(f"{name}: ok")
        except Exception:
            print(f"{name}: não configurado (tudo bem se não precisar)")
except ImportError:
    print("Fora do Kaggle: use variáveis de ambiente.")

## Candidatos e juiz

O **nome** à esquerda precisa ser igual ao `model` do `ml/eval/models.toml`. O **repositório** à direita é o id no Hugging Face — **confira** na célula seguinte: ela mostra se o modelo existe e qual licença ele declara. Os candidatos têm de ser **Apache-2.0 ou MIT** (uso comercial e uso das saídas liberados — `docs/DECISIONS.md` D-023).

In [ ]:
CANDIDATES = {
    # nome no models.toml -> repositório no Hugging Face (CONFIRA o id exato no site)
    "qwen35-4b": "Qwen/Qwen3.5-4B",
    "gemma4-e4b": "google/gemma-4-E4B-it",
    "tucano2-3.7b": "TucanoBR/Tucano2-qwen-3.7B-Instruct",
}
JUDGE = ("juiz", "Qwen/Qwen3-14B-AWQ")  # juiz: modelo aberto maior, Apache-2.0, quantizado para caber na T4

In [ ]:
# 3) Confere existência e licença de cada modelo (registre o resultado no docs/PROGRESS.md)
from huggingface_hub import model_info
for name, repo in [*CANDIDATES.items(), JUDGE]:
    try:
        info = model_info(repo, token=os.environ.get("HF_TOKEN"))
        license_tags = [t for t in info.tags if t.startswith("license:")]
        params = getattr(info, "safetensors", None)
        size = f"{params.total / 1e9:.1f}B parâmetros" if params and params.total else "tamanho ?"
        print(f"{name:14} {repo:45} {license_tags or 'licença não declarada!'}  {size}  gated={info.gated}")
    except Exception as err:
        print(f"{name:14} {repo:45} ERRO: {err.__class__.__name__}: {err}")

In [ ]:
# 4) Funções para subir/derrubar um modelo no vLLM
import subprocess, time, urllib.request

def serve(repo, served_name, port, extra=()):
    # T4 não tem bfloat16: usamos float16. Se um modelo gerar texto sem sentido em float16,
    # troque para "--dtype", "float32" (mais lento) e anote no relatório.
    cmd = ["vllm", "serve", repo, "--served-model-name", served_name, "--port", str(port),
           "--dtype", "half", "--max-model-len", "8192", "--gpu-memory-utilization", "0.85", *extra]
    log = open(f"/kaggle/working/vllm-{served_name}.log", "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):  # até 15 min para baixar e carregar
        time.sleep(5)
        if proc.poll() is not None:
            raise RuntimeError(f"vLLM parou. Veja /kaggle/working/vllm-{served_name}.log")
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=3)
            print(f"{served_name} no ar na porta {port}")
            return proc
        except Exception:
            pass
    proc.terminate()
    raise TimeoutError(f"{served_name} não subiu a tempo")

def stop(proc):
    proc.terminate()
    proc.wait(timeout=60)
    time.sleep(10)  # libera a memória da GPU

In [ ]:
# 5) Linha de base de cada candidato (ENEM + instruções + segurança).
#    Rodar de novo continua de onde parou (as respostas ficam em ml/eval/results/).
%cd /kaggle/working/a-aia
REGISTRY = {"qwen35-4b": "qwen35_4b", "gemma4-e4b": "gemma4_e4b", "tucano2-3.7b": "tucano2_3_7b"}
for served_name, repo in CANDIDATES.items():
    proc = serve(repo, served_name, 8000)
    try:
        !python ml/eval/run.py --model {REGISTRY[served_name]}
    finally:
        stop(proc)

In [ ]:
# 6) (Opcional) O Pro (Gemini) também, se PRO_API_KEY estiver nos Secrets
if os.environ.get("PRO_API_KEY"):
    !python ml/eval/run.py --model pro

In [ ]:
# 7) Juiz: modelo maior na porta 8001, julga as respostas de todos
# A T4 não suporta o kernel awq_marlin; o vLLM escolhe sozinho o kernel AWQ compatível.
judge_proc = serve(JUDGE[1], JUDGE[0], 8001)
try:
    for model in [*REGISTRY.values(), *(["pro"] if os.environ.get("PRO_API_KEY") else [])]:
        !python ml/eval/judge.py --model {model} --judge juiz
finally:
    stop(judge_proc)

In [ ]:
# 8) Resultados: relatórios + respostas brutas num zip para baixar (aba Output)
!ls ml/eval/reports
!cd ml/eval && zip -qr /kaggle/working/avaliacao.zip reports results
print("Baixe /kaggle/working/avaliacao.zip e copie a pasta reports/ para ml/eval/reports/ no seu repositório.")